# Триаж машинного перевода для постредактирования (QE), zh → ru

**Вариант 1**: предсказываем, насколько плох сегмент машинного перевода и нужно ли его вообще показывать человеку — **без эталонного перевода на инференсе**.

Пайплайн пилота:

1. **Данные** — берём zh-ru пары (UN Parallel Corpus).
2. **MT** — прогоняем китайскую сторону через готовую MT-модель (Helsinki-NLP/opus-mt-zh-ru).
3. **Авторазметка качества** — сравниваем MT-перевод с человеческим ru (chrF + LaBSE cosine similarity). Это делается **только на этапе сборки обучающих данных** — сам эталон в признаки не идёт.
4. **Признаки без эталона** — только из пары (src, mt): длины, jieba-токенизация, доля редких слов, NER-несоответствия, cross-lingual LaBSE similarity src↔mt.
5. **Статистика** — корреляция признаков с качеством, t-test между группами accept/edit.
6. **ML** — регрессия качества (0–1) + классификация accept/edit, ансамбли (RandomForest, GradientBoosting).

Ноутбук рассчитан на Google Colab. GPU (Runtime → Change runtime type → GPU) ускорит перевод и эмбеддинги, но не обязателен для пилотной выборки.

> `N_SAMPLES` ниже нарочно небольшой (для быстрой проверки, что пайплайн целиком работает). После проверки увеличьте его.


In [ ]:
!pip install -q jieba sacrebleu sentencepiece "transformers>=4.30" sentence-transformers datasets scikit-learn pandas numpy matplotlib scipy
!python -m spacy download ru_core_news_sm -q
!python -m spacy download zh_core_web_sm -q


## 1. Данные — сбор zh-ru корпуса

Пробуем загрузить UN Parallel Corpus через `datasets`. Если конфигурация недоступна (нет сети / поменялось имя датасета), падаем на небольшой встроенный набор — так ноутбук всё равно проходит целиком end-to-end для отладки пайплайна.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

N_SAMPLES = 300  # пилотная выборка; увеличьте после проверки пайплайна

try:
    from datasets import load_dataset
    raw = load_dataset("un_pc", "ru-zh", split="train", streaming=True)
    pairs = []
    for i, ex in enumerate(raw):
        if i >= N_SAMPLES:
            break
        t = ex["translation"]
        pairs.append({"zh": t["zh"], "ru": t["ru"]})
    df = pd.DataFrame(pairs)
    print(f"Загружено из UN Parallel Corpus: {len(df)} пар")
except Exception as e:
    print("Не удалось загрузить un_pc, использую запасной мини-набор:", repr(e))
    fallback = [
        {"zh": "联合国大会今天通过了一项新的决议。",
         "ru": "Генеральная Ассамблея Организации Объединенных Наций сегодня приняла новую резолюцию."},
        {"zh": "安全理事会将于下周举行紧急会议。",
         "ru": "Совет Безопасности проведет чрезвычайное заседание на следующей неделе."},
        {"zh": "各成员国应当遵守国际法的基本原则。",
         "ru": "Все государства-члены должны соблюдать основные принципы международного права."},
        {"zh": "秘书长呼吁各方保持克制，避免局势进一步升级。",
         "ru": "Генеральный секретарь призвал все стороны проявлять сдержанность и не допускать дальнейшей эскалации."},
        {"zh": "该报告涵盖了过去五年的发展情况。",
         "ru": "Этот доклад охватывает изменения за последние пять лет."},
        {"zh": "代表团对该决议草案表示强烈支持。",
         "ru": "Делегация заявила о своей решительной поддержке данного проекта резолюции."},
        {"zh": "会议讨论了气候变化对发展中国家的影响。",
         "ru": "На заседании обсуждалось воздействие изменения климата на развивающиеся страны."},
        {"zh": "预算委员会将审查明年的财政计划。",
         "ru": "Бюджетный комитет рассмотрит финансовый план на следующий год."},
        {"zh": "人权理事会通过了一项关于难民保护的决议。",
         "ru": "Совет по правам человека принял резолюцию о защите беженцев."},
        {"zh": "本组织致力于促进国际和平与安全。",
         "ru": "Организация стремится содействовать международному миру и безопасности."},
        {"zh": "各国代表团于周一抵达日内瓦参加谈判。",
         "ru": "Делегации разных стран прибыли в Женеву в понедельник для участия в переговорах."},
        {"zh": "该项目获得了多个国家的资金支持。",
         "ru": "Этот проект получил финансовую поддержку от нескольких стран."},
    ]
    df = pd.DataFrame(fallback)

df.head()


## 2. Генерация машинного перевода (MT)

Это и есть тот MT, качество которого мы будем оценивать (триажить).

In [ ]:
import torch
from transformers import pipeline

device = 0 if torch.cuda.is_available() else -1
mt_translator = pipeline("translation", model="Helsinki-NLP/opus-mt-zh-ru", device=device)

def translate_batch(texts, batch_size=16):
    outputs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        res = mt_translator(batch, max_length=256)
        outputs.extend([r["translation_text"] for r in res])
    return outputs

df["mt_ru"] = translate_batch(df["zh"].tolist())
df[["zh", "mt_ru", "ru"]].head()


## 3. Авторазметка качества (для обучения)

Комбинируем два сигнала близости MT к человеческому переводу:
- **chrF** — символьная n-грамная метрика (устойчива к морфологии ru и сегментации zh);
- **LaBSE cosine similarity** между эмбеддингами MT и эталона — семантическая близость.

Важно: `ru` (эталон) используется **только здесь**, чтобы получить `quality_score`/`needs_edit` для обучающей выборки. В признаках модели (шаг 4) эталон не участвует — на инференсе его не будет.


In [ ]:
import sacrebleu
from sentence_transformers import SentenceTransformer, util

def sentence_chrf(hyp, ref):
    return sacrebleu.sentence_chrf(hyp, [ref]).score / 100.0

df["chrf"] = [sentence_chrf(mt, ref) for mt, ref in zip(df["mt_ru"], df["ru"])]

labse = SentenceTransformer("sentence-transformers/LaBSE")
emb_mt_ref = labse.encode(df["mt_ru"].tolist(), convert_to_tensor=True, show_progress_bar=True)
emb_ref = labse.encode(df["ru"].tolist(), convert_to_tensor=True, show_progress_bar=True)
df["labse_sim_mt_ref"] = util.pairwise_cos_sim(emb_mt_ref, emb_ref).cpu().numpy()

df["quality_score"] = (df["chrf"] + df["labse_sim_mt_ref"]) / 2

EDIT_THRESHOLD = 0.6  # подберите порог под реальную практику постредактирования
df["needs_edit"] = (df["quality_score"] < EDIT_THRESHOLD).astype(int)

df[["zh", "mt_ru", "ru", "chrf", "labse_sim_mt_ref", "quality_score", "needs_edit"]].head()


## 4. Признаки без эталона

Всё, что ниже, считается только из `(zh, mt_ru)` — это в точности то, что доступно на инференсе для нового, ещё не отредактированного контента.


In [ ]:
import re
from collections import Counter

import jieba

# частотный словарь по самой пилотной выборке (для прод-системы лучше внешний частотный список zh)
all_zh_tokens = [tok for text in df["zh"] for tok in jieba.lcut(text)]
freq = Counter(all_zh_tokens)
RARE_THRESHOLD = 2

def zh_stats(text):
    tokens = jieba.lcut(text)
    n = len(tokens) or 1
    rare_ratio = sum(1 for t in tokens if freq[t] <= RARE_THRESHOLD) / n
    return len(text), n, rare_ratio

def count_digits(text):
    return len(re.findall(r"\d+", text))

rows = []
for _, r in df.iterrows():
    zh_chars, zh_tokens, zh_rare_ratio = zh_stats(r["zh"])
    mt_chars = len(r["mt_ru"])
    mt_tokens = len(r["mt_ru"].split())
    rows.append({
        "zh_len_chars": zh_chars,
        "zh_len_tokens": zh_tokens,
        "zh_rare_ratio": zh_rare_ratio,
        "mt_len_chars": mt_chars,
        "mt_len_tokens": mt_tokens,
        "len_ratio": mt_chars / (zh_chars or 1),
        "digit_mismatch": abs(count_digits(r["zh"]) - count_digits(r["mt_ru"])),
    })

feat_df = pd.DataFrame(rows)
feat_df.head()


In [ ]:
import spacy

nlp_zh = spacy.load("zh_core_web_sm")
nlp_ru = spacy.load("ru_core_news_sm")

def entity_counts(nlp, texts):
    return [len(doc.ents) for doc in nlp.pipe(texts)]

feat_df["zh_n_entities"] = entity_counts(nlp_zh, df["zh"].tolist())
feat_df["mt_n_entities"] = entity_counts(nlp_ru, df["mt_ru"].tolist())
feat_df["entity_mismatch"] = (feat_df["zh_n_entities"] - feat_df["mt_n_entities"]).abs()
feat_df.head()


In [ ]:
# Cross-lingual similarity src↔mt через LaBSE — БЕЗ эталона, валидно на инференсе
emb_src = labse.encode(df["zh"].tolist(), convert_to_tensor=True, show_progress_bar=True)
emb_mt = labse.encode(df["mt_ru"].tolist(), convert_to_tensor=True, show_progress_bar=True)
feat_df["labse_sim_src_mt"] = util.pairwise_cos_sim(emb_src, emb_mt).cpu().numpy()

feat_df["quality_score"] = df["quality_score"]
feat_df["needs_edit"] = df["needs_edit"]

feat_df.to_csv("features.csv", index=False)
print("Сохранено features.csv:", feat_df.shape)
feat_df.head()


## 5. Статистика — связь признаков с качеством

`features.csv` можно открыть и в R (`read.csv`) для параллельного анализа (`lm(quality_score ~ ., data = ...)`, ANOVA и т.п.) — это отдельный стат-блок проекта.


In [ ]:
from scipy import stats

feature_cols = [c for c in feat_df.columns if c not in ("quality_score", "needs_edit")]

corr_rows = []
for col in feature_cols:
    r, p = stats.pearsonr(feat_df[col], feat_df["quality_score"])
    corr_rows.append({"feature": col, "pearson_r": r, "p_value": p})

corr_df = pd.DataFrame(corr_rows).sort_values("pearson_r", key=lambda s: s.abs(), ascending=False)
corr_df


In [ ]:
# t-test: отличаются ли признаки между группами accept (needs_edit=0) и edit (needs_edit=1)
for col in feature_cols:
    good = feat_df.loc[feat_df["needs_edit"] == 0, col]
    bad = feat_df.loc[feat_df["needs_edit"] == 1, col]
    if len(good) > 1 and len(bad) > 1:
        t, p = stats.ttest_ind(good, bad, equal_var=False)
        print(f"{col:20s} t={t:6.2f}  p={p:.4f}")
    else:
        print(f"{col:20s} недостаточно данных в одной из групп (увеличьте N_SAMPLES)")


## 6. ML: регрессия качества + классификация accept/edit

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
)

X = feat_df[feature_cols]
y_reg = feat_df["quality_score"]
y_clf = feat_df["needs_edit"]

X_train, X_test, yreg_train, yreg_test, yclf_train, yclf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)


In [ ]:
reg_models = {
    "RandomForest": RandomForestRegressor(n_estimators=300, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(random_state=42),
}

for name, model in reg_models.items():
    model.fit(X_train, yreg_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(yreg_test, pred))
    mae = mean_absolute_error(yreg_test, pred)
    r, _ = stats.pearsonr(yreg_test, pred)
    print(f"{name:16s} RMSE={rmse:.3f}  MAE={mae:.3f}  Pearson r={r:.3f}")


In [ ]:
clf_models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced"),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
}

fitted_clf = {}
for name, model in clf_models.items():
    model.fit(X_train, yclf_train)
    fitted_clf[name] = model
    pred = model.predict(X_test)
    acc = accuracy_score(yclf_test, pred)
    f1 = f1_score(yclf_test, pred, zero_division=0)
    print(f"=== {name} === accuracy={acc:.3f}  F1={f1:.3f}")
    print(classification_report(yclf_test, pred, target_names=["accept", "needs_edit"], zero_division=0))


In [ ]:
import matplotlib.pyplot as plt

best_clf = fitted_clf["RandomForest"]
importances = pd.Series(best_clf.feature_importances_, index=feature_cols).sort_values()

plt.figure(figsize=(6, 4))
importances.plot(kind="barh")
plt.title("Важность признаков (RandomForest, accept/edit)")
plt.tight_layout()
plt.show()


## Итоги и следующие шаги

- `N_SAMPLES` в шаге 1 маленький специально для отладки — после того как весь пайплайн прошёл без ошибок, увеличьте выборку (тысячи пар) и переобучите модели.
- Признаки — стартовый набор. Логичные добавки: POS-несоответствия src/mt, синтаксическая сложность (глубина дерева зависимостей), type-token ratio, доля непереведённых китайских символов в MT (частый сбой MT).
- Порог `EDIT_THRESHOLD` в шаге 3 определяет метку `needs_edit` — его стоит откалибровать по реальным решениям переводчиков (accept/edit), а не выбирать произвольно.
- Для честной оценки QE-модели сравните её метрики с сегмент-левел корреляцией человеческих оценок качества, если/когда такие появятся (сейчас `quality_score` — прокси на основе chrF+LaBSE, а не человеческая оценка).
- Следующий шаг по канве проекта: R-скрипт со статистическим анализом `features.csv` (регрессия, ANOVA) как отдельный ноутбук/скрипт в этом репозитории.
